In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import precision_recall_curve

In [2]:
# Paths and configuration

CV_RESULTS_DIR = Path("../PH_q_LOGOCV_results")
MODEL_NAMES = ["ESM2", "ESMC", "LucaOne", "BacFormer", "AF3"]
Q_LIST = [0.999, 0.995, 0.990, 0.950, 0.900, 0.850, 0.800, 0.750]

OUT_DIR = Path("threshold_selection_results")
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

In [3]:
# All model-quantile combinations

model_quantile_df = pd.DataFrame(
    [
        {
            "model": model,
            "quantile": q,
        }
        for model in MODEL_NAMES
        for q in Q_LIST
    ]
)

model_quantile_df["quantile"] = (
    pd.to_numeric(
        model_quantile_df["quantile"],
        errors="coerce",
    )
    .round(3)
)

display(model_quantile_df)
print(f"Total model-quantile combinations: {len(model_quantile_df)}")

,model,quantile
0,ESM2,0.999
1,ESM2,0.995
2,ESM2,0.990
3,ESM2,0.950
4,ESM2,0.900
5,ESM2,0.850
6,ESM2,0.800
7,ESM2,0.750
8,ESMC,0.999
9,ESMC,0.995


Total model-quantile combinations: 40


In [4]:
# Calculating the exact F1-optimal threshold for every model-quantile combination

threshold_results = []
threshold_curves = {}
for row in model_quantile_df.itertuples(index=False):
    model, q = row.model, row.quantile
    pickle_file = (CV_RESULTS_DIR / f"{model}_results_raw" / f"logocv_both_unseen_results_{model}_q_{q:.3f}.pickle")
    if not pickle_file.exists():
        raise FileNotFoundError(pickle_file)

    with open(pickle_file, "rb") as f:
        result = pickle.load(f)

    labels = np.asarray(result["pooled_labels"], dtype=int)
    scores = np.asarray(result["pooled_scores"], dtype=float)
    precision, recall, thresholds = precision_recall_curve(labels, scores)

    # Last precision/recall point has no corresponding threshold
    precision = precision[:-1]
    recall = recall[:-1]

    f1 = (2 * precision * recall / (precision + recall + 1e-12))
    best_idx = np.argmax(f1)
    threshold_results.append(
        {
            "model": model,
            "quantile": q,
            "threshold": thresholds[best_idx],
            "precision": precision[best_idx],
            "recall": recall[best_idx],
            "f1": f1[best_idx],
            "n_labels": len(labels),
            "n_positives": int(labels.sum()),
        }
    )

    threshold_curves[(model, q)] = {
        "thresholds": thresholds,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

threshold_df = pd.DataFrame(threshold_results)
threshold_df.to_csv(OUT_DIR / "model_quantile_f1_thresholds.csv", index=False)
display(threshold_df.sort_values(["model", "quantile"]).round(3))

,model,quantile,threshold,precision,recall,f1,n_labels,n_positives
39,AF3,0.750,0.044,0.271,0.709,0.392,8954,1800
38,AF3,0.800,0.000,0.213,0.908,0.345,8954,1800
37,AF3,0.850,0.282,0.351,0.513,0.417,8954,1800
36,AF3,0.900,0.287,0.446,0.565,0.499,8954,1800
35,AF3,0.950,0.276,0.405,0.498,0.446,8954,1800
34,AF3,0.990,0.988,0.734,0.541,0.623,8954,1800
33,AF3,0.995,0.956,0.629,0.584,0.606,8954,1800
32,AF3,0.999,0.971,0.670,0.565,0.613,8954,1800
31,BacFormer,0.750,0.004,0.239,0.636,0.347,8954,1800
30,BacFormer,0.800,0.004,0.264,0.675,0.380,8954,1800


In [ ]:
# Display thresholds for the model-quantile combinations selected by routing

routing_df = pd.read_csv("../inference_n_inVitro_benchmark/both_unseen_inference_predictions_NEW/both_unseen_final_route_summary.csv")
selected_routes_df = (routing_df[["model", "pair_quantile_min"]]
    .rename(columns={"pair_quantile_min": "quantile"})
    .drop_duplicates()
)

selected_routes_df["quantile"] = (pd.to_numeric(selected_routes_df["quantile"], errors="coerce").round(3))
routed_threshold_df = (
    selected_routes_df
    .merge(
        threshold_df,
        on=["model", "quantile"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["model", "quantile"])
    .reset_index(drop=True)
)

display(routed_threshold_df.round(3))

,model,quantile,threshold,precision,recall,f1,n_labels,n_positives
0,AF3,0.75,0.044,0.271,0.709,0.392,8954,1800
1,AF3,0.85,0.282,0.351,0.513,0.417,8954,1800
2,AF3,0.90,0.287,0.446,0.565,0.499,8954,1800
3,AF3,0.95,0.276,0.405,0.498,0.446,8954,1800
4,BacFormer,0.80,0.004,0.264,0.675,0.380,8954,1800
5,BacFormer,0.90,0.088,0.371,0.448,0.406,8954,1800
6,ESM2,0.75,0.001,0.286,0.736,0.412,8954,1800
7,ESM2,0.80,0.113,0.325,0.497,0.393,8954,1800
8,ESM2,0.85,0.001,0.267,0.661,0.381,8954,1800
9,ESM2,0.90,0.032,0.311,0.530,0.392,8954,1800
